<a href="https://colab.research.google.com/github/Aham0803/PySpark/blob/main/Pyspark_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pyspark Data cleaning and handling missing values

### import the req libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum,count,when,col,lit,lower,trim,avg,coalesce

### creating a new spark session

In [3]:
spark = SparkSession.builder.appName("DataCleaning").getOrCreate()

In [4]:
## create mock dirty data

In [10]:
data = [
    (1, "Gourav", "Development", 1000000, "2023-01-01"),
    (2, "Ron", None, 500000, "2023-01-11"),
    (3, "Rohit", "Sales", None, "2023-05-12"),
    (4, "Amit", None, 800000, "2024-01-01"),
    (5, "Divya", "IT", 2000000, "2023-05-01"),
    (7, "Dev", "Marketing", -100, "2023-03-01"), ### Salary is Outlier
    (6, "Pranit", "Development", None, "2024-01-01"),
    (1, "Gourav", "Development", 1000000, "2023-01-01") ### Dublicated entry
]

schema =["id","Name" ,"department" ,"salary" ,"DateOfJoining"]

In [11]:
df = spark.createDataFrame(data,schema = schema)

In [12]:
df.show()

+---+------+-----------+-------+-------------+
| id|  Name| department| salary|DateOfJoining|
+---+------+-----------+-------+-------------+
|  1|Gourav|Development|1000000|   2023-01-01|
|  2|   Ron|       NULL| 500000|   2023-01-11|
|  3| Rohit|      Sales|   NULL|   2023-05-12|
|  4|  Amit|       NULL| 800000|   2024-01-01|
|  5| Divya|         IT|2000000|   2023-05-01|
|  7|   Dev|  Marketing|   -100|   2023-03-01|
|  6|Pranit|Development|   NULL|   2024-01-01|
|  1|Gourav|Development|1000000|   2023-01-01|
+---+------+-----------+-------+-------------+



## checking the null values

In [13]:
null_count = df.select([sum(when(col(c).isNull() ,1).otherwise(0)).alias(c) for c in df.columns])
null_count.show()

+---+----+----------+------+-------------+
| id|Name|department|salary|DateOfJoining|
+---+----+----------+------+-------------+
|  0|   0|         2|     2|            0|
+---+----+----------+------+-------------+



## Droping the duplicates entries

In [15]:
df_cleaned = df.drop_duplicates()

In [16]:
df_cleaned.show()

+---+------+-----------+-------+-------------+
| id|  Name| department| salary|DateOfJoining|
+---+------+-----------+-------+-------------+
|  1|Gourav|Development|1000000|   2023-01-01|
|  7|   Dev|  Marketing|   -100|   2023-03-01|
|  5| Divya|         IT|2000000|   2023-05-01|
|  3| Rohit|      Sales|   NULL|   2023-05-12|
|  4|  Amit|       NULL| 800000|   2024-01-01|
|  2|   Ron|       NULL| 500000|   2023-01-11|
|  6|Pranit|Development|   NULL|   2024-01-01|
+---+------+-----------+-------+-------------+



## dropping null values on the basis of relevant coulmn

In [25]:
df_dropped = df_cleaned.dropna(subset=["id" ,"Name"])

In [26]:
df_dropped.show()

+---+------+-----------+-------+-------------+
| id|  Name| department| salary|DateOfJoining|
+---+------+-----------+-------+-------------+
|  1|Gourav|Development|1000000|   2023-01-01|
|  7|   Dev|  Marketing|   -100|   2023-03-01|
|  5| Divya|         IT|2000000|   2023-05-01|
|  3| Rohit|      Sales|   NULL|   2023-05-12|
|  4|  Amit|       NULL| 800000|   2024-01-01|
|  2|   Ron|       NULL| 500000|   2023-01-11|
|  6|Pranit|Development|   NULL|   2024-01-01|
+---+------+-----------+-------+-------------+



## handling missing values through data imputation

In [27]:
df_filled = df_dropped.fillna({
    "department":"NoDeptSelected",
    "salary": 0
})
df_filled.show()

+---+------+--------------+-------+-------------+
| id|  Name|    department| salary|DateOfJoining|
+---+------+--------------+-------+-------------+
|  1|Gourav|   Development|1000000|   2023-01-01|
|  7|   Dev|     Marketing|   -100|   2023-03-01|
|  5| Divya|            IT|2000000|   2023-05-01|
|  3| Rohit|         Sales|      0|   2023-05-12|
|  4|  Amit|NoDeptSelected| 800000|   2024-01-01|
|  2|   Ron|NoDeptSelected| 500000|   2023-01-11|
|  6|Pranit|   Development|      0|   2024-01-01|
+---+------+--------------+-------+-------------+



In [29]:
avg_salary = df_filled.select(avg("salary")).collect()[0][0]
df_imputed = df_filled.withColumn("salary" , when(col("salary") == 0 , avg_salary).otherwise(col("salary")))
df_imputed.show()

+---+------+--------------+-----------------+-------------+
| id|  Name|    department|           salary|DateOfJoining|
+---+------+--------------+-----------------+-------------+
|  1|Gourav|   Development|        1000000.0|   2023-01-01|
|  7|   Dev|     Marketing|           -100.0|   2023-03-01|
|  5| Divya|            IT|        2000000.0|   2023-05-01|
|  3| Rohit|         Sales|614271.4285714285|   2023-05-12|
|  4|  Amit|NoDeptSelected|         800000.0|   2024-01-01|
|  2|   Ron|NoDeptSelected|         500000.0|   2023-01-11|
|  6|Pranit|   Development|614271.4285714285|   2024-01-01|
+---+------+--------------+-----------------+-------------+

